In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch
import torch.nn as nn
import math

class Transpose(nn.Module):
    def __init__(self, dim0, dim1):
        super().__init__()
        self.dim0 = dim0
        self.dim1 = dim1

    def forward(self, x):
        return x.transpose(self.dim0, self.dim1)

class PositionalEncoding2D(nn.Module):
    def __init__(self, num_patches, dim):
        super().__init__()
        self.register_buffer('pos_embed', self.build_sincos_encoding(num_patches, dim), persistent=False)

    def build_sincos_encoding(self, num_patches, dim):
        pe = torch.zeros(num_patches, dim)
        position = torch.arange(0, num_patches, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # [1, num_patches, dim]

    def forward(self, x):
        return x + self.pos_embed[:, :x.size(1), :]

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.GroupNorm(8, out_channels)
        )
        self.skip = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        return self.block(x) + self.skip(x)

class CondEncoder(nn.Module):
    def __init__(self, in_channels=4, out_channels=736, num_tokens=64):
        super().__init__()
        self.encoder = nn.Sequential(
            ResidualBlock(in_channels, 64), # [B, 64, 64, 64]
            nn.AvgPool2d(2), # [B, 64, 32, 32]
            ResidualBlock(64, 128),
            nn.AvgPool2d(2), # [B, 128, 16, 16]
            ResidualBlock(128, 256),
            nn.AvgPool2d(2), # [B, 256, 8, 8]
            nn.Conv2d(256, out_channels, kernel_size=1) # [B, 736, 8, 8]
        )
        self.proj = nn.Sequential(
            nn.Flatten(2),  # [B, 736, 64]
            Transpose(-1, -2),   # [B, 64, 736]
        )
        self.pos_embed = PositionalEncoding2D(num_patches=num_tokens, dim=out_channels)
        self.norm = nn.LayerNorm(out_channels)

    def forward(self, x):
        feat = self.encoder(x)          # [B, 736, 8, 8]
        tokens = self.proj(feat)        # [B, 64, 736]
        tokens = self.pos_embed(tokens) # [B, 64, 736]
        tokens = self.norm(tokens)
        return tokens


In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from diffusers import DDPMScheduler, UNet2DConditionModel, AutoencoderKL
from accelerate import Accelerator
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

class OutpaintTrainer:
    def __init__(self, pretrained_path="runwayml/stable-diffusion-v1-5"):
        # Initialize Accelerator for FP16 mixed precision
        self.accelerator = Accelerator(mixed_precision='fp16')
        # Load model components
        self.vae = AutoencoderKL.from_pretrained(pretrained_path, subfolder="vae")
        self.unet = UNet2DConditionModel.from_pretrained(pretrained_path, subfolder="unet")

        # Coordinate Encoder
        self.coord_encoder = nn.Sequential(nn.Linear(4, 32), nn.GELU())

        # Conditional Projection Layer

        self.cond_proj = CondEncoder()

        # Prepare components for mixed precision
        components = [self.vae, self.unet, self.coord_encoder, self.cond_proj]
        self.vae, self.unet, self.coord_encoder, self.cond_proj = \
            self.accelerator.prepare(*components)

        # Optimizer
        self.optimizer = torch.optim.AdamW(
            list(self.unet.parameters()) +
            list(self.coord_encoder.parameters()) +
            list(self.cond_proj.parameters()),
            lr=5e-5)
        self.optimizer = self.accelerator.prepare(self.optimizer)

        # Noise Scheduler
        self.noise_scheduler = DDPMScheduler.from_pretrained(
            pretrained_path, subfolder="scheduler")

        # Freeze VAE
        self.vae.requires_grad_(False)
        self.accelerator.register_for_checkpointing(self.unet, self.coord_encoder, self.cond_proj, self.optimizer)

In [5]:
import torch.nn.functional as F

def gaussian_blur_rgb(img, k=41, sigma=10.0):
    """Separable Gaussian blur on RGB. img in [-1,1], (B,3,H,W)"""
    pad = k // 2
    x = torch.arange(k, device=img.device, dtype=img.dtype) - pad
    g = torch.exp(-(x**2) / (2 * sigma**2))
    g = g / g.sum()
    gx = g.view(1,1,1,k).repeat(3,1,1,1)
    gy = g.view(1,1,k,1).repeat(3,1,1,1)
    img = F.pad(img, (pad,pad,0,0), mode='reflect')
    img = F.conv2d(img, gx, groups=3)
    img = F.pad(img, (0,0,pad,pad), mode='reflect')
    img = F.conv2d(img, gy, groups=3)
    return img

def build_color_map(image_tensor, fg_mask=None):
    """
    Build a low-frequency color map; protect background if fg_mask provided.
    image_tensor: (B,3,512,512) in [-1,1]
    fg_mask: optional (B,1,512,512) in [0,1]
    """
    cmap = gaussian_blur_rgb(image_tensor, k=41, sigma=10.0)
    if fg_mask is not None:
        # keep original image on background to avoid tinting whitespace
        cmap = fg_mask * cmap + (1.0 - fg_mask) * image_tensor
    return cmap

def cosine_taper(step_idx, total_steps, start_frac=0.0, end_frac=0.35):
    """Returns a weight in [0,1] that starts at 1 and decays to 0 over [start_frac, end_frac] of steps."""
    import math
    s = int(total_steps * start_frac)
    e = max(s + 1, int(total_steps * end_frac))
    if step_idx < s or step_idx > e:
        return 0.0
    x = (step_idx - s) / (e - s)
    return 0.5 * (1.0 + math.cos(math.pi * x))  # 1 -> 0

# --- 2) Tiny Color Encoder ----------------------------------------------------
class ColorEncoder(nn.Module):
    """
    Encodes a smoothed RGB map to 64 spatial tokens (8x8) of dimension token_dim.
    Keep intentionally tiny to carry only low-frequency color/layout.
    """
    def __init__(self, token_dim=128):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d((8, 8))      # 512x512 -> 8x8
        self.proj = nn.Linear(3, token_dim)           # per (R,G,B) -> token
        self.register_buffer("pe", self._build_sincos_pe(8, 8, token_dim), persistent=False)

    @staticmethod
    def _build_sincos_pe(h, w, dim):
        # simple 2D sincos positional encodings
        y = torch.arange(h).float().unsqueeze(1).repeat(1, w)
        x = torch.arange(w).float().unsqueeze(0).repeat(h, 1)
        xy = torch.stack([x, y], dim=-1)  # (h, w, 2)
        feats = []
        for i in range(max(1, dim // 4)):
            feats.append(torch.sin(xy / (10000 ** (2 * i / dim))))
            feats.append(torch.cos(xy / (10000 ** (2 * i / dim))))
        pe = torch.cat(feats, dim=-1)  # (h, w, ~dim)
        if pe.shape[-1] < dim:
            pad = dim - pe.shape[-1]
            pe = torch.cat([pe, torch.zeros(h, w, pad)], dim=-1)
        return pe  # (h, w, dim)

    def forward(self, rgb_smooth):  # (B,3,H,W) in [-1,1]
        B = rgb_smooth.shape[0]
        x = self.pool(rgb_smooth)                 # (B,3,8,8)
        x = x.permute(0, 2, 3, 1).contiguous()    # (B,8,8,3)
        x = self.proj(x)                          # (B,8,8,token_dim)
        pe = self.pe.to(x.device).unsqueeze(0)    # (1,8,8,token_dim)
        x = x + pe
        tokens = x.view(B, 64, -1)                # (B,64,token_dim)
        return tokens

In [6]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
from PIL import Image

class OutpaintEngine:
    def __init__(self, device="cuda"):
        self.device = device

        # These must be loaded externally:
        self.vae = None
        self.unet = None
        self.accelerator = None
        self.noise_scheduler = None
        self.coord_encoder = None
        self.cond_proj = None

        self.directions = ['right', 'left', 'down', 'up']

        self.transform = transforms.Compose([
            transforms.Resize((512, 512)),
            transforms.ToTensor(),
            transforms.Normalize([0.5] * 3, [0.5] * 3)
        ])

        ##COLOR ENCODERS INIT
        self.color_encoder = ColorEncoder(token_dim=128).to(self.device)
        self.color_token_scale = 1.0   # try 0.5–1.5
        self.color_t_start = 0.0       # first 0–35% steps
        self.color_t_end   = 0.35

        self.cross_dim = 768  # will be corrected by init_heads() if needed

        # If your base context (cond_proj + coord_encoder) already sums to 768,
        # you can leave base_fuse as identity.
        self.base_fuse  = nn.Identity().to(self.device)

        # Project color tokens (128) -> cross_dim (usually 768)
        self.color_proj = nn.Linear(128, self.cross_dim).to(self.device)

    def _create_latent_mask(self, bbox, latent_shape):
        """
        Create latent mask for the diffusion process based on bbox.
        bbox shape: (B, 4) = (x1, y1, x2, y2) in normalized coords
        latent_shape: (B, C, H, W)
        """
        b, _, lh, lw = latent_shape
        masks = []
        for coords in bbox:
            x1 = coords[0] * lw
            y1 = coords[1] * lh
            x2 = coords[2] * lw
            y2 = coords[3] * lh

            xx, yy = torch.meshgrid(
                torch.arange(lw, device=self.device),
                torch.arange(lh, device=self.device)
            )
            mask = ((xx >= x1) & (xx <= x2) &
                    (yy >= y1) & (yy <= y2)).float()
            masks.append(mask)
        return torch.stack(masks).unsqueeze(1)

    def generate_iterative(self, image_tensor, steps=200, iterations=10, fg_mask_img=None):
        """
        Iteratively fill a vertical gap in the center of an image
        by generating the central 64-pixel-wide region each time,
        shifting left/right in latent space, and filling the gap with white latent.
        """

        # Initial stitched image
        current_image = image_tensor.clone()
        b, c, h, w = current_image.shape

        # Define the central gap bbox in normalized coords
        bbox = torch.tensor([[0.0, 0.4375, 1.0, 0.5625]], device=self.device)

        # Initially encode to latent space
        with torch.no_grad():
            current_latent = self.vae.encode(current_image).latent_dist.sample()
            current_latent = current_latent * self.vae.config.scaling_factor

        # BUILD SMOOTHED COLOR MAP
        # If you have a soft foreground mask (B,1,512,512) in [0,1], pass it;
        # otherwise leave fg_mask_img=None to skip background protection.
        color_map = build_color_map(current_image, fg_mask=fg_mask_img)        # (B,3,512,512)
        with torch.no_grad():
          color_tokens_736 = self.color_encoder(color_map)

        for i in range(iterations):
            print(f"[{i+1}/{iterations}] Filling central gap in latent space")

            # Step 1: Create latent mask
            latent_mask = self._create_latent_mask(bbox, current_latent.shape)

            # Step 2: Masked latent
            masked_latent = current_latent * (1 - latent_mask)

            # Step 3: Add noise
            noise = torch.randn_like(current_latent)
            noisy_latent = self.noise_scheduler.add_noise(
                masked_latent * latent_mask,
                noise * latent_mask,
                torch.tensor(steps)
            )
            noisy_latent = masked_latent * (1 - latent_mask) + noisy_latent * latent_mask

            # Step 4: Denoising loop + Color noise additions
            self.noise_scheduler.set_timesteps(steps)
            latent_input = noisy_latent

            condition = torch.cat([
                self.cond_proj(masked_latent),
                self.coord_encoder(bbox).unsqueeze(1).expand(-1, 64, -1)
            ], dim=-1)
            cnt = 0
            for t in self.noise_scheduler.timesteps:
                cnt += 1
                # Keep unmasked region clamped to context latent
                latent_input = latent_input * latent_mask + masked_latent * (1 - latent_mask)

                # ### ⬇ COLOR GUIDANCE: schedule + pack tokens
                w_col = cosine_taper(cnt, steps, self.color_t_start, self.color_t_end) * self.color_token_scale
                if w_col > 0:
                    col_tokens = color_tokens_736 * w_col           # (B,64,128)
                else:
                    col_tokens = torch.zeros_like(color_tokens_736) # (B,64,128)

                # Base tokens you already use
                # base tokens (your existing two sources)
                base_tokens  = self.cond_proj(masked_latent)                            # (B, 64, D1)
                coord_tokens = self.coord_encoder(bbox).unsqueeze(1).expand(-1,64,-1)   # (B, 64, D2)
                base_cat     = torch.cat([base_tokens, coord_tokens], dim=-1)           # (B, 64, 768)
                base_ctx     = self.base_fuse(base_cat)                                 # (B, 64, 768)

                # color tokens path
                w_col = cosine_taper(cnt, steps, self.color_t_start, self.color_t_end) * self.color_token_scale
                if w_col > 0:
                    color_ctx = self.color_proj(color_tokens_736) * w_col               # (B, 64, 768)
                else:
                    color_ctx = torch.zeros_like(base_ctx)[:, :64]                       # (B, 64, 768)

                # final encoder_hidden_states: concat along SEQUENCE dim
                condition = torch.cat([base_ctx, color_ctx], dim=1)                     # (B, 128, 768)
                """

                # Final conditioning
                condition = torch.cat([base_tokens, coord_tokens, col_tokens], dim=-1)   # (B,64,D1+D2+128)"""

                with torch.no_grad():
                    noise_pred = self.unet(latent_input, t, encoder_hidden_states=condition).sample

                latent_input = self.noise_scheduler.step(noise_pred, t, latent_input).prev_sample

                # Optional preview near the end
                if cnt == steps - 1:
                    with torch.no_grad():
                        generated_latent = latent_input * latent_mask + masked_latent * (1 - latent_mask)
                        generated_img = self.vae.decode(generated_latent / self.vae.config.scaling_factor).sample
                        preview = ((generated_img[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5).clip(0, 1) * 255).astype(np.uint8)
                        Image.fromarray(preview).save(f"{t}.png")
                        plt.figure(figsize=(6,6)); plt.imshow(preview); plt.axis("off")

            # --- at the very end of generate_iterative ---
            if i == iterations - 1:
                with torch.no_grad():
                    generated_latent = latent_input * latent_mask + masked_latent * (1 - latent_mask)
                    generated_img = self.vae.decode(generated_latent / self.vae.config.scaling_factor).sample
                    preview = ((generated_img[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
                              .clip(0, 1) * 255).astype(np.uint8)

                # return decoded preview (optional) AND raw latent tensor on CPU
                return preview, generated_latent.cpu()


                      # Step 5: merge latent
            # extract the newly generated center region from latent_input
            new_patch_latent = latent_input[..., :, :, 28:36]   # latent pixels [28:36] = 64 image pixels

            # split into left and right halves (each 4 latent pixels)
            left_latent_patch = new_patch_latent[..., :, :, :4]   # (1, C, H, 4)
            right_latent_patch = new_patch_latent[..., :, :, 4:]  # (1, C, H, 4)

            # crop sides from current latent
            cropped_latent = current_latent[..., :, :, 4:-4]      # remove 4 latent pixels from each side → shape (1, C, H, 56)

            # split cropped latent into left and right parts
            cropped_left = cropped_latent[..., :, :, :24]         # left part
            cropped_right = cropped_latent[..., :, :, -24:]       # right part

            # create white gap latent (8 latent pixels = 64 image pixels)
            white_gap_latent = torch.zeros((1, current_latent.shape[1], current_latent.shape[2], 8), device=self.device)

            # concatenate parts:
            # cropped_left | left_patch | white_gap | right_patch | cropped_right
            stitched_latent = torch.cat([
                cropped_left,
                left_latent_patch,
                white_gap_latent,
                right_latent_patch,
                cropped_right
            ], dim=-1)

            # Crop back to 64 latent pixels width
            start = (stitched_latent.shape[-1] - 64) // 2
            stitched_latent = stitched_latent[..., :, :, start:start+64]

            # Update for next iteration
            current_latent = stitched_latent

            # decode and show intermediate result
            with torch.no_grad():
                decoded_img = self.vae.decode(current_latent / self.vae.config.scaling_factor).sample

            preview = (decoded_img[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5).clip(0, 1)
            Image.fromarray((preview * 255).astype(np.uint8)).save(f"iteration_{i+1}.png")

            plt.figure(figsize=(6,6))
            plt.imshow(preview)
            plt.title(f"Iteration {i+1}")
            plt.axis("off")
            plt.show()




In [7]:
trainer = OutpaintTrainer()
trainer.accelerator.load_state("drive/MyDrive/checkpoint_LR")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

vae/diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

In [8]:
import os
from PIL import Image
import numpy as np

# 假设 outpaint_engine 和 trainer 已经初始化
outpaint_engine = OutpaintEngine()
outpaint_engine.unet = trainer.unet
outpaint_engine.cond_proj = trainer.cond_proj
outpaint_engine.coord_encoder = trainer.coord_encoder
outpaint_engine.vae = trainer.vae
outpaint_engine.accelerator = trainer.accelerator
outpaint_engine.noise_scheduler = trainer.noise_scheduler

# 所有 step 数及对应后缀
step_suffix_list = [(150, "1150"), (200, "1200"), (250, "1250")]

import os
from PIL import Image
import torch

input_folder = "drive/MyDrive/DISCO_USECASES/202509GapFillerOldResults/filltest"
latent_dir   = "drive/MyDrive/20251006latent_eval/20251017GapFillerColorGuidedLatents/"
os.makedirs(latent_dir, exist_ok=True)

# One pass per image, one latent saved
for filename in sorted(os.listdir(input_folder)):
    if not filename.lower().endswith(".png"):
        continue

    filepath  = os.path.join(input_folder, filename)
    base_name = os.path.splitext(filename)[0]   # e.g., "pair_18_15_leftright"

    img = Image.open(filepath).convert("RGB")
    img_tensor = outpaint_engine.transform(img).unsqueeze(0).to(outpaint_engine.accelerator.device)

    # Generate once (adjust steps/iterations as you like)
    preview_uint8, latent_tensor = outpaint_engine.generate_iterative(
        img_tensor,
        steps=200,
        iterations=6
    )

    # Save exactly one latent per input, matching the image name
    latent_pt_path = os.path.join(latent_dir, f"{base_name}_latent.pt")

    # Optional: avoid accidental overwrite
    # if os.path.exists(latent_pt_path):
    #     print(f"[SKIP] {latent_pt_path} already exists.")
    #     continue

    torch.save(latent_tensor, latent_pt_path)
    print(f"[Saved] {latent_pt_path}")

print("Done — one *_latent.pt per input image saved.")



Output hidden; open in https://colab.research.google.com to view.